# 02 — Structural Analysis (Data Mining Stage)

We compute structural metrics on all attribution graphs and analyze the relationship between graph structure
and interpretability.

**Steps:**
1. Load all attribution graphs
2. Compute structural metrics for each graph
3. Analyze metric distributions by graph type
4. Correlation analysis
5. Statistical significance testing
6. Key findings and visualizations

In [1]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path

from src.graph_generator import AttributionGraph
from src.structural_metrics import compute_all_metrics, compute_metrics_batch, StructuralMetrics
from src.dataset import load_graphs_from_dir
from src.utils import (
    plot_metric_distributions,
    plot_correlation_matrix,
    plot_metric_vs_interpretability,
    generate_metrics_summary,
    compare_graph_types,
)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

## 1. Load All Graphs

In [2]:
# Load synthetic graphs
synthetic_graphs = load_graphs_from_dir('../data/raw/synthetic')
print(f'Synthetic graphs: {len(synthetic_graphs)}')

# Load real graphs (if generated)
real_graphs = []
categories = [
    'factual_recall', 'reasoning', 'creative_writing',
    'code_understanding', 'ambiguous_context', 'multilingual',
]
for cat in categories:
    cat_dir = Path(f'../data/raw/{cat}')
    if cat_dir.exists():
        cat_graphs = load_graphs_from_dir(str(cat_dir))
        for g in cat_graphs:
            g.metadata['category'] = cat
        real_graphs.extend(cat_graphs)

print(f'Real graphs: {len(real_graphs)}')
print(f'Total: {len(synthetic_graphs) + len(real_graphs)}')

Loaded 375 graphs from ../data/raw/synthetic
Synthetic graphs: 375
Loaded 5 graphs from ../data/raw/factual_recall
Loaded 5 graphs from ../data/raw/reasoning
Loaded 0 graphs from ../data/raw/creative_writing
Loaded 0 graphs from ../data/raw/code_understanding
Loaded 0 graphs from ../data/raw/ambiguous_context
Loaded 0 graphs from ../data/raw/multilingual
Real graphs: 10
Total: 385


## 2. Compute Structural Metrics

In [3]:
print('Computing metrics for synthetic graphs...')
synthetic_metrics = compute_metrics_batch(synthetic_graphs)

if real_graphs:
    print('\nComputing metrics for real graphs...')
    real_metrics = compute_metrics_batch(real_graphs)
else:
    real_metrics = []

all_metrics = synthetic_metrics + real_metrics
print(f'\nTotal metrics computed: {len(all_metrics)}')

Computing metrics for synthetic graphs...


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/networkx/algorithms/assortativity/correlation.py:302: RuntimeWarning: invalid value encountered in scalar divide
  return float((xy * (M - ab)).sum() / np.sqrt(vara * varb))
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/networkx/algorithms/assortativity/correlation.py:302: RuntimeWarning: invalid value encountered in sqrt
  return float((xy * (M - ab)).sum() / np.sqrt(vara * varb))


  Computed metrics for 50/375 graphs
  Computed metrics for 100/375 graphs
  Computed metrics for 150/375 graphs
  Computed metrics for 200/375 graphs
  Computed metrics for 250/375 graphs
  Computed metrics for 300/375 graphs
  Computed metrics for 350/375 graphs

Computing metrics for real graphs...

Total metrics computed: 385


In [4]:
df = pd.DataFrame([m.to_dict() for m in all_metrics])
print(f'DataFrame shape: {df.shape}')
df.head()

DataFrame shape: (385, 37)


,prompt,graph_id,n_nodes,n_edges,density,mean_in_degree,std_in_degree,max_in_degree,mean_out_degree,std_out_degree,...,backward_edge_ratio,layer_entropy,dag_depth,tree_likeness,mean_activation,std_activation,mean_edge_weight,std_edge_weight,weight_entropy,interpretability_label
0,[synthetic-clean-tree],graph_0000,104,103,0.009615,0.990385,1.496761,4,0.990385,0.097585,...,1.0,0.907125,3.206286,1.0,0.139773,0.102982,0.646761,0.227926,4.571052,1.0
1,[synthetic-clean-tree],graph_0001,76,75,0.013158,0.986842,1.400126,4,0.986842,0.113951,...,1.0,0.977276,3.326596,1.0,0.155711,0.130712,0.646097,0.216244,4.258928,1.0
2,[synthetic-clean-tree],graph_0002,86,85,0.011628,0.988372,1.505758,4,0.988372,0.107204,...,1.0,0.924267,3.294978,1.0,0.152156,0.123713,0.638217,0.199453,4.391921,1.0
3,[synthetic-clean-tree],graph_0003,156,155,0.006410,0.993590,1.491415,4,0.993590,0.079807,...,1.0,0.907435,3.544764,1.0,0.144469,0.105115,0.656632,0.212824,4.987779,1.0
4,[synthetic-clean-tree],graph_0004,124,123,0.008065,0.991935,1.478316,4,0.991935,0.089440,...,1.0,0.924021,3.600073,1.0,0.128611,0.106738,0.670327,0.217453,4.756722,1.0


## 3. Metric Distributions by Graph Type

Our core hypothesis: interpretable circuits (clean trees) have structurally
different properties than uninterpretable ones (tangled graphs).

In [5]:
plot_metric_distributions(synthetic_metrics, save_dir='../results/figures')

Saved metric distributions to ../results/figures/metric_distributions.png


In [6]:
comparison = compare_graph_types(synthetic_metrics)
print('Mean Structural Metrics by Graph Type:')
print('=' * 70)
comparison

Mean Structural Metrics by Graph Type:


type,Interpretable,Mixed,Uninterpretable
n_nodes,1.253600e+02,44.746667,50.000000
n_edges,1.243600e+02,127.800000,369.400000
density,8.924932e-03,0.065757,0.150776
mean_in_degree,9.910751e-01,2.859059,7.388000
std_in_degree,1.485662e+00,3.186961,2.466917
max_in_degree,4.000000e+00,10.706667,13.346667
mean_out_degree,9.910751e-01,2.859059,7.388000
std_out_degree,9.264923e-02,1.692580,2.458821
max_out_degree,1.000000e+00,7.426667,13.453333
degree_assortativity,6.926553e-10,0.316026,-0.017503


## 4. Correlation Analysis

Which structural metrics are most correlated with interpretability?

In [7]:
plot_correlation_matrix(synthetic_metrics, save_dir='../results/figures')

Saved correlation matrix to ../results/figures/correlation_matrix.png


In [8]:
plot_metric_vs_interpretability(synthetic_metrics, save_dir='../results/figures', top_n=6)

/Users/owenhughes/Library/Python/3.13/lib/python/site-packages/numpy/lib/_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/owenhughes/Library/Python/3.13/lib/python/site-packages/numpy/lib/_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Saved scatter plots to ../results/figures/metric_vs_interpretability.png


In [9]:
df_synth = pd.DataFrame([m.to_dict() for m in synthetic_metrics])
numeric_cols = df_synth.select_dtypes(include=[np.number]).columns
feature_cols = [c for c in numeric_cols if c not in ['interpretability_label', 'n_nodes', 'n_edges']]

correlations = df_synth[feature_cols].corrwith(df_synth['interpretability_label'])
ranked = correlations.abs().sort_values(ascending=False)

print('Metrics Ranked by Correlation with Interpretability:')
print('=' * 55)
for feat, corr in ranked.head(15).items():
    direction = '+' if correlations[feat] > 0 else '-'
    print(f'  {feat:35s}  |r| = {corr:.4f}  ({direction})')

Metrics Ranked by Correlation with Interpretability:
  modularity                           |r| = 0.9929  (+)
  density                              |r| = 0.9917  (-)
  max_out_degree                       |r| = 0.9873  (-)
  diameter                             |r| = 0.9872  (+)
  avg_clustering                       |r| = 0.9867  (-)
  backward_edge_ratio                  |r| = 0.9802  (+)
  mean_out_degree                      |r| = 0.9797  (-)
  mean_in_degree                       |r| = 0.9797  (-)
  std_activation                       |r| = 0.9770  (-)
  std_out_degree                       |r| = 0.9762  (-)
  tree_likeness                        |r| = 0.9742  (+)
  mean_activation                      |r| = 0.9696  (-)
  avg_shortest_path                    |r| = 0.9641  (+)
  max_in_degree                        |r| = 0.9495  (-)
  cross_layer_edge_ratio               |r| = 0.9262  (-)


/Users/owenhughes/Library/Python/3.13/lib/python/site-packages/numpy/lib/_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/owenhughes/Library/Python/3.13/lib/python/site-packages/numpy/lib/_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


## 5. Statistical Significance Testing

For the top metrics, test whether the difference between
interpretable and uninterpretable graphs is statistically significant.

In [10]:
df_synth = pd.DataFrame([m.to_dict() for m in synthetic_metrics])
interpretable = df_synth[df_synth['interpretability_label'] == 1.0]
uninterpretable = df_synth[df_synth['interpretability_label'] == 0.0]

print('Statistical Significance Tests (Welch t-test):')
print('=' * 70)
print(f'{"Metric":35s} {"t-stat":>10s} {"p-value":>12s} {"Significant":>12s}')
print('-' * 70)

for feat in ranked.head(10).index:
    t_stat, p_val = stats.ttest_ind(
        interpretable[feat].dropna(),
        uninterpretable[feat].dropna(),
        equal_var=False,
    )
    sig = 'YES ***' if p_val < 0.001 else ('YES **' if p_val < 0.01 else ('YES *' if p_val < 0.05 else 'no'))
    print(f'{feat:35s} {t_stat:10.3f} {p_val:12.2e} {sig:>12s}')

Statistical Significance Tests (Welch t-test):
Metric                                  t-stat      p-value  Significant
----------------------------------------------------------------------
modularity                             192.750    3.46e-252      YES ***
density                               -213.344    9.88e-239      YES ***
max_out_degree                        -126.920    9.89e-154      YES ***
diameter                                   inf     0.00e+00      YES ***
avg_clustering                        -208.830    9.08e-186      YES ***
backward_edge_ratio                    286.826    2.98e-206      YES ***
mean_out_degree                       -213.256    3.83e-187      YES ***
mean_in_degree                        -213.256    3.83e-187      YES ***
std_activation                         -85.753    2.42e-207      YES ***
std_out_degree                        -125.402    4.29e-154      YES ***


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


## 6. Key Findings

In [11]:
# Save metrics to CSV
df_all = pd.DataFrame([m.to_dict() for m in all_metrics])
df_all.to_csv('../results/metrics/structural_metrics.csv', index=False)
print(f'Saved {len(df_all)} rows to results/metrics/structural_metrics.csv')

summary = generate_metrics_summary(all_metrics)
summary.to_csv('../results/metrics/metrics_summary.csv')
print(f'Saved summary statistics to results/metrics/metrics_summary.csv')

Saved 385 rows to results/metrics/structural_metrics.csv
Saved summary statistics to results/metrics/metrics_summary.csv
